# 🏛️ LexAgent v3.0: Multi-Agent Legal Debate Framework
### **Caselaw Access Project (CAP) Primary Database Edition — [OFFLINE RUN / UNLIMITED HIGH-END GPU]**

Welcome to the **LexAgent v3.0** master execution notebook on branch `offline-run`.
In this edition, all sample limits have been removed, full 30-case benchmark and ablation evaluations are enabled by default, and high-end Nvidia GPU acceleration (A100 / H100 / RTX 4090 / A6000) with bfloat16 precision and offline cluster support is pre-configured.

### Supported Capabilities in this Notebook:
1. **Offline & High-End GPU Guide**: Detailed reference of configurable parameters across `config.py` and `master_run.py`.
2. **Environment Setup**: Local NVMe SSD directory configuration, dependency validation, and GPU verification.
3. **Step 1: Scalable Multi-Volume DB Construction**: Ingest CAP volumes with parallel multi-threading and checkpointing (landmark, modern, all, or custom range).
4. **Step 2: Run Individual or All Phases**:
   - **Phase 1**: Hybrid Retrieval alone.
   - **Phase 2**: Fixed 1-Round Debate with Evidence-Disjoint Defense (Novelty 3).
   - **Phase 3**: Full DVA+ Dynamic Debate (Factorized CCE Novelty 1 + DDC + Memory).
5. **Step 3: Full 30-Case Benchmark (No Limits)**: Execute full evaluation suite with scorecards (HR, CAS, EOR, DCR, Confidence).
6. **Step 4: Full 30-Case Ablation Studies (No Limits)**: Run 5 component ablation variants with comparative analysis.
7. **Step 5: Citation Stress-Test Protocol**: Zero-LLM controlled diagnostic test across all 6 citation classes (Novelty 5).


---
# ⚡ Offline Server & High-End Nvidia GPU Configuration Guide

This branch (`offline-run`) is specifically optimized for running on a standalone or air-gapped server equipped with top-tier Nvidia GPUs (such as **A100 (40/80GB)**, **H100 (80GB)**, **RTX 4090 (24GB)**, **RTX 3090**, or **RTX A6000 (48GB)**).

### 1. Environment Variables You Can Modify (Shell / `.bashrc` / Terminal)
Set these variables before executing `master_run.py` or running notebook cells:

| Environment Variable | Recommended Value | Description |
|---|---|---|
| `LEXAGENT_PRECISION` | `bfloat16` or `float16` | Runs Mistral-7B in **native unquantized 16-bit** without NF4 4-bit quantization, unlocking faster inference and peak legal generation quality on 24GB+ GPUs. |
| `TRANSFORMERS_OFFLINE` | `1` | Forces Hugging Face transformers to use locally downloaded model files without attempting internet connections. |
| `HF_HUB_OFFLINE` | `1` | Forces Hugging Face Hub snapshot loader to strictly read from local cache. |
| `LEXAGENT_DATA_DIR` | `/path/to/fast/nvme/data` | Overrides the default data root to your high-speed local NVMe scratch disk. |
| `LEXAGENT_BATCH_SIZE` | `256` or `512` | Batch size for Legal-BERT dense vector embedding into ChromaDB (blazing fast on A100/H100). |
| `LEXAGENT_WORKERS` | `16` or `32` | Number of parallel worker threads for multi-volume CAP downloading and chunk parsing. |
| `LEXAGENT_CCE_SCAN_LIMIT` | `8000` | Deepens the NLI Cross-Encoder opinion text window scan depth (up to 8,000 characters per case). |
| `LEXAGENT_MAX_ROUNDS` | `3` | Maximum dynamic adversarial debate rounds allowed before Judge synthesis. |
| `CUDA_VISIBLE_DEVICES` | `0` (or `0,1`) | Target GPU device index for multi-GPU servers. |

```bash
# Example setup script for offline server run:
export CUDA_VISIBLE_DEVICES=0
export LEXAGENT_PRECISION=bfloat16
export LEXAGENT_BATCH_SIZE=256
export LEXAGENT_WORKERS=16
export LEXAGENT_CCE_SCAN_LIMIT=8000
# export TRANSFORMERS_OFFLINE=1  # Uncomment if server is air-gapped without internet
# export HF_HUB_OFFLINE=1        # Uncomment if server is air-gapped without internet
```

---
### 2. Things You Can Modify in `config.py`

1. **Model Weights & Local Paths**:
   - `MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"` -> Change to a local directory path (e.g. `"/models/Mistral-7B-Instruct-v0.3"`) if weights are already pre-downloaded on disk.
   - `EMBEDDER_NAME = "nlpaueb/legal-bert-base-uncased"` -> Local directory path for Legal-BERT weights.
   - `NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"` -> Can upgrade to `"cross-encoder/nli-deberta-v3-base"` or `"cross-encoder/nli-deberta-v3-large"` on high-end GPUs for even higher entailment precision.
2. **CAP Volume Scope** (`CAP_VOLUMES_MODE`):
   - `"landmark"`: 27 landmark constitutional volumes covering all 30 benchmark cases (~10 mins to build).
   - `"modern"`: 285 volumes from Vol 300 to 585 (~1940 to 2018), covering the modern SCOTUS era.
   - `"all"`: Full 600+ volumes of U.S. Reports from Vol 1 to present.
3. **Thresholds & Verification Weights**:
   - `TAU_ACCEPTANCE_THRESHOLD = 0.70`: Conformal acceptance threshold $\tau$ for the factorized CCE engine.
   - `CCS_VERIFIED_THRESHOLD = 0.75` and `CCS_UNCERTAIN_THRESHOLD = 0.40`: Classification boundaries for the 5-way verdict (`SUPPORTED`, `UNCERTAIN`, `MISATTRIBUTED`, `FABRICATED`, `UNVERIFIABLE`).
   - `DDC_CCS_CONTINUE_THRESH = 0.60`: If average citation confidence falls below 0.60, the Dynamic Debate Controller triggers another round of counter-retrieval.

---
### 3. Things You Can Modify in `master_run.py` & CLI Flags

| CLI Argument | Offline Server Recommendation | Function |
|---|---|---|
| `--volumes` | `landmark`, `modern`, or `all` | Selects volume subset to ingest into ChromaDB and BM25. |
| `--benchmark-limit` | `30` | Number of benchmark cases to evaluate (set to `30` for full unlimited run). |
| `--ablation-limit` | `30` | Number of cases per ablation variant (all 5 graph topologies evaluated over 30 cases). |
| `--workers` | `16` or `32` | Number of CPU worker threads for parallel volume downloading. |
| `--batch-size` | `256` or `512` | Batch size for GPU embedding ingestion into ChromaDB. |
| `--portion-size` | `50` or `100` | Number of volumes processed per disk checkpoint. |
| `--stress-test` | *(standalone)* | Runs zero-LLM CCE diagnostic gate over all 6 citation classes without sample limits. |
| `--baseline` | *(standalone)* | Runs standard single-agent RAG baseline over all 30 cases for direct empirical comparison. |
| `--all` | *(standalone)* | End-to-end execution: Build DB -> Smoke Tests -> Stress Test -> Full Benchmark -> Full Ablations. |


---
# ⚡ Offline Server & High-End Nvidia GPU Configuration Guide

This branch (`offline-run`) is specifically optimized for running on a standalone or air-gapped server equipped with top-tier Nvidia GPUs (such as **A100 (40/80GB)**, **H100 (80GB)**, **RTX 4090 (24GB)**, **RTX 3090**, or **RTX A6000 (48GB)**).

### 1. Environment Variables You Can Modify (Shell / `.bashrc` / Terminal)
Set these variables before executing `master_run.py` or running notebook cells:

| Environment Variable | Recommended Value | Description |
|---|---|---|
| `LEXAGENT_PRECISION` | `bfloat16` or `float16` | Runs Mistral-7B in **native unquantized 16-bit** without NF4 4-bit quantization, unlocking faster inference and peak legal generation quality on 24GB+ GPUs. |
| `TRANSFORMERS_OFFLINE` | `1` | Forces Hugging Face transformers to use locally downloaded model files without attempting internet connections. |
| `HF_HUB_OFFLINE` | `1` | Forces Hugging Face Hub snapshot loader to strictly read from local cache. |
| `LEXAGENT_DATA_DIR` | `/path/to/fast/nvme/data` | Overrides the default data root to your high-speed local NVMe scratch disk. |
| `LEXAGENT_BATCH_SIZE` | `256` or `512` | Batch size for Legal-BERT dense vector embedding into ChromaDB (blazing fast on A100/H100). |
| `LEXAGENT_WORKERS` | `16` or `32` | Number of parallel worker threads for multi-volume CAP downloading and chunk parsing. |
| `LEXAGENT_CCE_SCAN_LIMIT` | `8000` | Deepens the NLI Cross-Encoder opinion text window scan depth (up to 8,000 characters per case). |
| `LEXAGENT_MAX_ROUNDS` | `3` | Maximum dynamic adversarial debate rounds allowed before Judge synthesis. |
| `CUDA_VISIBLE_DEVICES` | `0` (or `0,1`) | Target GPU device index for multi-GPU servers. |

```bash
# Example setup script for offline server run:
export CUDA_VISIBLE_DEVICES=0
export LEXAGENT_PRECISION=bfloat16
export LEXAGENT_BATCH_SIZE=256
export LEXAGENT_WORKERS=16
export LEXAGENT_CCE_SCAN_LIMIT=8000
# export TRANSFORMERS_OFFLINE=1  # Uncomment if server is air-gapped without internet
# export HF_HUB_OFFLINE=1        # Uncomment if server is air-gapped without internet
```

---
### 2. Things You Can Modify in `config.py`

1. **Model Weights & Local Paths**:
   - `MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"` -> Change to a local directory path (e.g. `"/models/Mistral-7B-Instruct-v0.3"`) if weights are already pre-downloaded on disk.
   - `EMBEDDER_NAME = "nlpaueb/legal-bert-base-uncased"` -> Local directory path for Legal-BERT weights.
   - `NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"` -> Can upgrade to `"cross-encoder/nli-deberta-v3-base"` or `"cross-encoder/nli-deberta-v3-large"` on high-end GPUs for even higher entailment precision.
2. **CAP Volume Scope** (`CAP_VOLUMES_MODE`):
   - `"landmark"`: 27 landmark constitutional volumes covering all 30 benchmark cases (~10 mins to build).
   - `"modern"`: 285 volumes from Vol 300 to 585 (~1940 to 2018), covering the modern SCOTUS era.
   - `"all"`: Full 600+ volumes of U.S. Reports from Vol 1 to present.
3. **Thresholds & Verification Weights**:
   - `TAU_ACCEPTANCE_THRESHOLD = 0.70`: Conformal acceptance threshold $\tau$ for the factorized CCE engine.
   - `CCS_VERIFIED_THRESHOLD = 0.75` and `CCS_UNCERTAIN_THRESHOLD = 0.40`: Classification boundaries for the 5-way verdict (`SUPPORTED`, `UNCERTAIN`, `MISATTRIBUTED`, `FABRICATED`, `UNVERIFIABLE`).
   - `DDC_CCS_CONTINUE_THRESH = 0.60`: If average citation confidence falls below 0.60, the Dynamic Debate Controller triggers another round of counter-retrieval.

---
### 3. Things You Can Modify in `master_run.py` & CLI Flags

| CLI Argument | Offline Server Recommendation | Function |
|---|---|---|
| `--volumes` | `landmark`, `modern`, or `all` | Selects volume subset to ingest into ChromaDB and BM25. |
| `--benchmark-limit` | `30` | Number of benchmark cases to evaluate (set to `30` for full unlimited run). |
| `--ablation-limit` | `30` | Number of cases per ablation variant (all 5 graph topologies evaluated over 30 cases). |
| `--workers` | `16` or `32` | Number of CPU worker threads for parallel volume downloading. |
| `--batch-size` | `256` or `512` | Batch size for GPU embedding ingestion into ChromaDB. |
| `--portion-size` | `50` or `100` | Number of volumes processed per disk checkpoint. |
| `--stress-test` | *(standalone)* | Runs zero-LLM CCE diagnostic gate over all 6 citation classes without sample limits. |
| `--baseline` | *(standalone)* | Runs standard single-agent RAG baseline over all 30 cases for direct empirical comparison. |
| `--all` | *(standalone)* | End-to-end execution: Build DB -> Smoke Tests -> Stress Test -> Full Benchmark -> Full Ablations. |


---
## 📦 Section 1: Environment Setup & Hardware Check

In [2]:
# 1.1 Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/V3_with_CAP_as_main
except Exception:
    print("Running in local environment.")

Mounted at /content/drive
/content/drive/MyDrive/V3_with_CAP_as_main


In [3]:
# 1.2 Install Required Dependencies
!pip install -q -r requirements.txt
!pip install -q chromadb rank-bm25 sentence-transformers fuzzywuzzy python-Levenshtein \
    transformers bitsandbytes accelerate langchain-community langgraph datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.

In [3]:
# 1.3 Hardware Sanity Check (T4 GPU Verification)
!nvidia-smi
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

Thu Sep 24 16:05:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 🛠️ Section 2: Step 1 — Build Primary CAP Authority Database
Supports multi-volume parallel ingestion with checkpointing.
Choose from:
- `--volumes landmark`: 27 key constitutional volumes (~10 mins, recommended)
- `--volumes modern`: Volumes 300 to 585 (~1940 to 2018)
- `--volumes all`: Full U.S. Reports (Volumes 1 to 600+)
- Or a custom range: `--volumes 400-450`

In [ ]:
# 2. Clear any stale Python cache
#!find /content/drive/MyDrive/V3_with_CAP_as_main -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null

# 3. Verify the import works
#!python -c "from runs.build_db import extract_doc_volume; print('✓ extract_doc_volume ready!')"

# 4. Run Step 1 database build
#!python /content/drive/MyDrive/V3_with_CAP_as_main/master_run.py --build-db --volumes landmark --workers 16 --batch-size 256


In [ ]:
# Build Database with Landmark Volumes Preset (or change --volumes to 'modern' / 'all')
#!python master_run.py --build-db --volumes landmark --workers 16 --batch-size 256

---
## 🔬 Section 3: Step 2 — Run Individual or All Phases
You can run each phase individually with a custom question, or run all phases sequentially.

In [5]:
# 3.1 Run Phase 1 Alone (Hybrid Retrieval over CAP)
!python master_run.py --phase 1 --query "Does the Fourth Amendment protect against warrantless cell phone searches?"


██████████████████████████████████████████████████████████████████████████████
  RUNNING PHASE 1: HYBRID RETRIEVAL OVER CAP DATABASE
██████████████████████████████████████████████████████████████████████████████

  PHASE 1 SMOKE TEST: CAP HYBRID RETRIEVAL (Dense + BM25)
  Query: 'Does the Fourth Amendment protect against warrantless cell phone searches?'
[LexAgent-V3] 2026-09-27 03:39:06 | src.models.load_model | INFO | Loading embedder: nlpaueb/legal-bert-base-uncased (on cuda)...
config.json: 100% 1.02k/1.02k [00:00<00:00, 2.77MB/s]

pytorch_model.bin: downloading bytes:  50% 219M/440M [00:01<00:00, 251MB/s, 16.9MB/s  ]
pytorch_model.bin: reconstructing file:  15% 67.1M/440M [00:02<00:11, 32.9MB/s]
pytorch_model.bin: downloading bytes:  72% 319M/440M [00:02<00:00, 220MB/s, 26.5MB/s  ]
pytorch_model.bin: downloading bytes:  94% 414M/440M [00:02<00:00, 260MB/s, 33.4MB/s  ]
pytorch_model.bin: downloading bytes: 100% 415M/415M [00:03<00:00, 121MB/s, 36.5MB/s  ]
pytorch_model.bin: recons

In [4]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"==================================================")
print(f" Pre-downloading {model_id}")
print(f" Resumes from the ~3.2 GB already fetched...")
print(f"==================================================")

# 1. Download model weights and configuration files
local_path = snapshot_download(
    repo_id=model_id,
    ignore_patterns=["*.onnx*", "*.msgpack*"],
    resume_download=True,
)

# 2. Pre-cache the tokenizer
AutoTokenizer.from_pretrained(model_id)

print(f"\n✓ Mistral-7B is fully downloaded and ready at:")
print(f"  {local_path}")


 Pre-downloading mistralai/Mistral-7B-Instruct-v0.3
 Resumes from the ~3.2 GB already fetched...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]


✓ Mistral-7B is fully downloaded and ready at:
  /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/c170c708c41dac9275d15a8fff4eca08d52bab71


In [6]:
# 3.2 Run Phase 2 Alone (Fixed 1-Round Adversarial Debate)
!python master_run.py --phase 2 --query "Does the Fourth Amendment protect against warrantless cell phone searches?"


██████████████████████████████████████████████████████████████████████████████
  RUNNING PHASE 2: FIXED 1-ROUND ADVERSARIAL DEBATE
██████████████████████████████████████████████████████████████████████████████

  PHASE 2 SMOKE TEST: FIXED 1-ROUND DEBATE (PROSECUTOR & DEFENSE)
  Query: 'Does the Fourth Amendment protect against warrantless cell phone searches?'
[LexAgent-V3] 2026-09-27 03:42:50 | src.models.load_model | INFO | Loading embedder: nlpaueb/legal-bert-base-uncased (on cuda)...
Loading weights: 100% 199/199 [00:00<00:00, 21789.45it/s]
[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
c

In [7]:
# 3.3 Run Phase 3 Alone (Full DVA+ Protocol with Active CCE & DDC)
!python master_run.py --phase 3 --query "Does the Fourth Amendment protect against warrantless cell phone searches?"


██████████████████████████████████████████████████████████████████████████████
  RUNNING PHASE 3: FULL DVA+ DYNAMIC DEBATE (CCE + DDC + MEMORY)
██████████████████████████████████████████████████████████████████████████████

  PHASE 3 SMOKE TEST: FULL DVA+ PIPELINE (CCE + DDC + MEMORY)
  Query: 'Does the Fourth Amendment protect against warrantless cell phone searches?'
[LexAgent-V3] 2026-09-27 03:47:37 | src.models.load_model | INFO | Loading embedder: nlpaueb/legal-bert-base-uncased (on cuda)...
Loading weights: 100% 199/199 [00:00<00:00, 16217.82it/s]
[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTE

In [ ]:
# 3.4 Run All Phases (Phase 1 -> 2 -> 3) Sequentially
!python master_run.py --test-phases

---
## 📊 Section 4: Step 3 — Comprehensive 30 Cases (Full Run) Benchmark
Evaluates the framework across the 25–30 legal questions with per-case agent breakdowns and aggregate metrics.

In [8]:
# Run Benchmark (up to 30 cases)
!python master_run.py --benchmark --benchmark-limit 30


██████████████████████████████████████████████████████████████████████████████
  STEP 3: RUNNING 25-CASE COMPREHENSIVE BENCHMARK EVALUATION
██████████████████████████████████████████████████████████████████████████████

  LEXAGENT v3.0 (CAP EDITION) — 25-CASE BENCHMARK EVALUATION
  Loaded 14524 canonical CAP cases and citations.
Loading weights: 100% 199/199 [00:00<00:00, 24086.41it/s]
Loading weights: 100% 291/291 [00:58<00:00,  4.94it/s]
/content/drive/MyDrive/V3_with_CAP_as_main/experiments/../src/models/load_model.py:111: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm_pipeline = HuggingFacePipeline(pipeline=text_pipeline)
Loading weights: 100% 106/106 [00:00<00:00, 832.73

---
## 🧪 Section 5: Step 4 — Run Ablation Studies
Evaluates the 4 component ablation variants against the full LexAgent v3.0 framework.

In [11]:
# Run Ablation Studies (10 cases per variant)
!python master_run.py --ablation --ablation-limit 30

^C


---
## 🎯 Section 6: Step 5 — Citation Stress-Test Protocol (Novelty 5)
Zero-LLM controlled diagnostic benchmark across 6 citation classes:
1. **Correct** (Target case + true holding) -> Expected: SUPPORTED
2. **Misattributed** (Target case + wrong case holding) -> Expected: MISATTRIBUTED
3. **CaseHOLD Distractor** (Target case + hard negative holding) -> Expected: MISATTRIBUTED
4. **Fabricated with Cite** (Invented name in covered volume) -> Expected: FABRICATED
5. **Fabricated without Cite** (Invented name, no reporter) -> Expected: UNVERIFIABLE
6. **Real Off-Corpus** (Landmark cases outside 27 volumes) -> Expected: UNVERIFIABLE (never fake)

In [10]:
# Run Citation Stress-Test Protocol (Fast, zero LLM calls, ~30 seconds)
!python master_run.py --stress-test


██████████████████████████████████████████████████████████████████████████████
  STEP 4: RUNNING CITATION STRESS-TEST PROTOCOL (Novelty 5 Diagnostic Gate)
██████████████████████████████████████████████████████████████████████████████

  LEXAGENT v3.0: CITATION STRESS-TEST PROTOCOL (Novelty 5)
  Controlled Diagnostic Verification Across 6 Error Types
[LexAgent-V3] 2026-09-27 05:39:07 | src.models.load_model | INFO | Loading embedder: nlpaueb/legal-bert-base-uncased (on cuda)...
Loading weights: 100% 199/199 [00:00<00:00, 19616.13it/s]
[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_rela

---
## 🚀 Section 7: Run Complete End-to-End Suite
Executes DB Construction -> Phase Smoke Tests -> Benchmark -> Ablations -> Stress Test in one go.

In [ ]:
!python master_run.py --all --volumes landmark --benchmark-limit 30 --ablation-limit 30